# 02. 메시지 생성 파이프라인 (논문 2장, 3.1~3.2절, 그림 1·2)

```
출석 JSON ─┬─ 파이프라인 1: 이미지 URL → Gemma-3n 멀티모달 → 숙제 완료 / 미완료
           └─ 파이프라인 2: MCP Tool(main2.py) → YouTube 인기 영상 제목 목록 → random.choice
                    ↓
       프롬프트(도구 결과 + 완료 여부 + 인증 정보 + 가이드라인) → Gemma → 동기부여 메시지
```

이 노트북의 메시지 생성은 **LoRA 학습 이전** 모델로 수행한 기존 프롬프트(30단어 이내) 결과입니다.
개선 프롬프트와 LoRA 모델 결과는 `03_improvement.ipynb`에서 이어집니다.

In [ ]:
# 필요 설치 라이브러리
!pip install -q transformers datasets peft accelerate bitsandbytes
!pip install -q datasets transformers peft accelerate bitsandbytes scikit-learn
!pip install transformers torch peft mcp accelerate bitsandbytes
!pip install nest_asyncio
!pip install --upgrade accelerate
!pip install --upgrade torch torchvision torchaudio transformers

## 1. 입력: 출석(숙제 인증) JSON

`id`, `content`(연속 인증 정보), `image_1`(인증 이미지 파일명)을 사용합니다. 공개 저장소에서는 학생 식별 정보를 예시 값으로 바꾸었습니다.

In [ ]:
user_command = input("명령을 입력하세요: ")
# ex) {"id":"sample_001","course_id":"<course_id>","member_id":"<member_id>","type":"attendance","content":"이번 챌린지 54회차 인증 완료<br \\/>(누적 4회 인증)","image_1":"sample_homework.jpg","image_2":null,"image_3":null,"image_4":null,"image_5":null,"image_6":null,"image_7":null,"image_8":null,"image_9":null,"image_10":null,"updated_at":"2025-07-29 00:09:56","created_at":"2025-07-29 00:09:55","is_del":"1"}

명령을 입력하세요: {"id":"sample_001","course_id":"<course_id>","member_id":"<member_id>","type":"attendance","content":"이번 챌린지 54회차 인증 완료<br \\/>(누적 4회 인증)","image_1":"sample_homework.jpg","image_2":null,"image_3":null,"image_4":null,"image_5":null,"image_6":null,"image_7":null,"image_8":null,"image_9":null,"image_10":null,"updated_at":"2025-07-29 00:09:56","created_at":"2025-07-29 00:09:55","is_del":"1"}


## 2. 파이프라인 1 — 숙제 인증 이미지 판별

`{id}`, `{image_1}`로 이미지 URL을 만들고, Gemma-3n-E2B-it(`AutoProcessor` + `Gemma3nForConditionalGeneration`)이 **완료 / 미완료**만 출력하도록 합니다.
결과는 `result_attendence`에 저장되어 메시지 방향(칭찬 / 격려)을 정합니다.

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, Gemma3nForConditionalGeneration
from PIL import Image
import requests
from io import BytesIO
from pathlib import Path
import json
# --- 1. 모델 ID 및 로컬 경로 설정 ---
# 이 경로는 멀티모달 Gemma 모델이 저장된 실제 경로여야 합니다. 경로를 확인하여 주세요
model_path = Path("/content/drive/MyDrive/gemma-3n-E2b-it") # 해당 부분에서 경로를 멀티모달 모델에 맞게 조정

print(f"\nGoogle 드라이브의 로컬 경로에서 멀티모달 Gemma 모델을 로드합니다: {model_path}...")

try:
    mm_processor = AutoProcessor.from_pretrained(model_path, local_files_only=True)
    mm_model = Gemma3nForConditionalGeneration.from_pretrained(
        model_path,
        torch_dtype="auto",
        device_map="auto",
        local_files_only=True
    ).eval()
    print("멀티모달 Gemma 모델이 Google 드라이브에서 성공적으로 로드되었습니다! 🎉")
except Exception as e:
    print(f"모델 로드 중 최종 오류 발생: {e}")
    print("새롭게 다운로드된 모델 파일에 문제가 있거나, GPU 메모리가 여전히 부족할 수 있습니다.")

# ------ 4. 이미지 로드 함수 -------
def load_image_from_url(url: str) -> Image.Image:

    "URL에서 이미지를 로드하여 PIL.Image 객체로 반환."
    try:
        response = requests.get(url)
        response.raise_for_status()
        image = Image.open(BytesIO(response.content)).convert("RGB")
        return image
    except requests.exceptions.RequestException as e:
        print(f"Error loading image from {url}: {e}")
        return None

# --- 5. 이미지 인식 및 텍스트 생성 함수 ---
def get_image_description(image_url: str, prompt_text: str) -> str:
    "이미지 URL과 텍스트 프롬프트를 사용하여 Gemma 모델로부터 설명을 생성합니다."

    print(f"\n--- 이미지를 분석합니다: {image_url} (프롬프트: '{prompt_text}') ---")

    image = load_image_from_url(image_url)

    if image is None:
        return f"이미지를 불러오는 데 실패했습니다: {image_url}. URL을 확인하거나 접근 권한을 확인해 주세요."

    # Gemma 모델에 전달할 메시지 리스트 구성
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt_text},
            ]
        }
    ]
    inputs = mm_processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(mm_model.device, dtype=torch.float16) # 모델과 동일한 디바이스, 데이터 타입으로 이동

    input_len = inputs["input_ids"].shape[-1]

    print("Gemma 응답을 생성 중...")
    with torch.inference_mode():
        outputs = mm_model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            pixel_values=inputs.get("pixel_values", None),
            audio_features=None,
            max_new_tokens=128,
            temperature=0.7,
            top_p=0.9,
            do_sample=False
        )


        generation = outputs[0][input_len:]

    decoded = mm_processor.decode(generation, skip_special_tokens=True)

    # 모델의 출력에서 실제 답변 부분만 추출 (Gemma의 채팅 형식에 맞춰 파싱)
    if "<start_of_turn>model" in decoded:
        response_start_index = decoded.find("<start_of_turn>model") + len("<start_of_turn>model")
        response_end_index = decoded.find("<end_of_turn>", response_start_index)
        if response_end_index != -1:
            final_response = decoded[response_start_index:response_end_index].strip()
        else:
            final_response = decoded[response_start_index:].strip()
    else:
        final_response = decoded.strip()

    return final_response

def process_attendance_data(attendance_json_str: str):
    """
    출석 데이터 JSON 문자열을 입력받아 숙제 이미지 URL을 생성하고,
    Gemma 모델을 사용하여 숙제 완료 여부를 판단합니다.
    """
    try:
        attendance_data = json.loads(attendance_json_str)
        attendance_id = attendance_data.get("id")
        filename = attendance_data.get("image_1")

        if attendance_id is None or filename is None:
            return "출석 데이터에 'id' 또는 'image_1' 필드가 없습니다."

        filename_without_ext = Path(filename).stem
        image_url = f"https://s3.ap-northeast-2.amazonaws.com/<bucket>/attendance/{attendance_id}/{filename_without_ext}.jpg"

        print(f"생성된 이미지 URL: {image_url}")

        prompt_homework = (
          "당신은 학생의 숙제 인증 이미지를 분석하는 AI입니다. "
          "이미지를 보고 숙제가 완료되었는지, 아니면 미완료 상태인지 판단하세요."
          "다른 문장, 설명, 기호는 절대 출력하지 말 것."
        )
        print(f"숙제 판단 프롬프트: '{prompt_homework}'")

        homework_completion_response = get_image_description(image_url, prompt_homework)
        return homework_completion_response

    except json.JSONDecodeError:
        return "유효하지 않은 JSON 형식입니다."

attendance_data_str = user_command

print("\n--- 숙제 완료 여부 판단 시작 ---")
result_attendence = process_attendance_data(attendance_data_str)
print(f"\n[최종 숙제 완료 판단 결과]:\n{result_attendence}")


Google 드라이브의 로컬 경로에서 멀티모달 Gemma 모델을 로드합니다: /content/drive/MyDrive/gemma-3n-E2b-it...


멀티모달 Gemma 모델이 Google 드라이브에서 성공적으로 로드되었습니다! 🎉

--- 숙제 완료 여부 판단 시작 ---
생성된 이미지 URL: https://s3.ap-northeast-2.amazonaws.com/<bucket>/attendance/sample_001/sample_homework.jpg
숙제 판단 프롬프트: '당신은 학생의 숙제 인증 이미지를 분석하는 AI입니다. 이미지를 보고 숙제가 완료되었는지, 아니면 미완료 상태인지 판단하세요.다른 문장, 설명, 기호는 절대 출력하지 말 것.'

--- 이미지를 분석합니다: https://s3.ap-northeast-2.amazonaws.com/<bucket>/attendance/sample_001/sample_homework.jpg (프롬프트: '당신은 학생의 숙제 인증 이미지를 분석하는 AI입니다. 이미지를 보고 숙제가 완료되었는지, 아니면 미완료 상태인지 판단하세요.다른 문장, 설명, 기호는 절대 출력하지 말 것.') ---


Gemma 응답을 생성 중...

[최종 숙제 완료 판단 결과]:
완료


## 3. 파이프라인 2 — MCP Tool로 실시간 콘텐츠 수집

`mcp-upgrade/main2.py`가 MCP 클라이언트 세션으로 `youtube_trend` / `youtube_music` 도구를 호출하고, YouTube Data API v3의 인기 급상승 영상 목록을 반환합니다.
반환된 목록을 제목 단위로 나눈 `result_tool`에서 `random.choice`로 하나를 골라 프롬프트에 넣습니다.

In [ ]:
from pathlib import Path

print("✅ 초기 설정 및 라이브러리 임포트 완료!")
# 경로 설정 바꾸기
model_path = Path("/content/drive/MyDrive/gemma-3n-E2b-it")
mcp_base_path = "/content/drive/MyDrive/MCP/mcp-upgrade/" # MCP 프로젝트 폴더 (mcp와 mcp_server.py를 포함하는) 경로

# Colab 노트북의 현재 작업 디렉토리를 MCP 프로젝트 폴더로 변경
%cd {mcp_base_path}
print(f"✅ 현재 작업 디렉토리가 '{mcp_base_path}'로 변경되었습니다.")
print(f"\nGoogle 드라이브의 로컬 경로에서 멀티모달 Gemma 모델을 로드할 준비 중: {model_path}...")
print(f"\nGoogle 드라이브의 로컬 경로에서 멀티모달 Gemma 모델을 로드합니다: {model_path}...")


✅ 초기 설정 및 라이브러리 임포트 완료!
/content/drive/MyDrive/MCP/mcp-upgrade
✅ 현재 작업 디렉토리가 '/content/drive/MyDrive/MCP/mcp-upgrade/'로 변경되었습니다.

Google 드라이브의 로컬 경로에서 멀티모달 Gemma 모델을 로드할 준비 중: /content/drive/MyDrive/gemma-3n-E2b-it...

Google 드라이브의 로컬 경로에서 멀티모달 Gemma 모델을 로드합니다: /content/drive/MyDrive/gemma-3n-E2b-it...


In [ ]:
# main2.py를 실행: mcp를 사용하여 최신 기반 데이터 받아오기

import subprocess, json, os
import random
import json
import ast
import re

SCRIPT = "/content/drive/MyDrive/MCP/mcp-upgrade/main2.py"

# 결과(stdout)만 문자열로 받아오기
out = subprocess.check_output(
    ["python", SCRIPT, "--just-tool"],
    text=True  # bytes → str
)
print("RAW:", out)

##str 형식을 list로 가져오기
data_dict = ast.literal_eval(out)
list_string_content = data_dict.get('text')
lines_split_by_number = re.split(r'\n+\d+\.\s*', list_string_content)
result_tool = []

if lines_split_by_number:
        # 첫 번째 항목은 split 결과의 첫 부분에 해당합니다.
        first_item = lines_split_by_number[0].strip()
        if first_item:
            result_tool.append(first_item)

        # 나머지 항목들은 split 패턴에 의해 번호가 제거된 상태로 추가됩니다.
        for item_content in lines_split_by_number[1:]:
            cleaned_item = item_content.strip()
            if cleaned_item:
                result_tool.append(cleaned_item)

print(random.choice(result_tool))
print(random.choice(result_tool))
print(random.choice(result_tool))

RAW: {'text': "1. 임영웅 'ULSSIGU' M/V (Performance Ver.) (임영웅)\n\n\n2. NMIXX(엔믹스) “Blue Valentine” Trailer: The Paradox of Love (JYP Entertainment)\n\n\n3. [최신가요 실시간 인기차트] 2025년 9월 27일 4주차, 멜론차트 X, 종합차트, 차트둥이 노래모음 KPOP 플레이리스트 (Beautiful Life)\n\n\n4. [MV] WOODZ(우즈) _ I'll Never Love Again (1theK (원더케이))\n\n\n5. 에스쿱스X민규 (SEVENTEEN) '5, 4, 3 (Pretty woman) (feat. Lay Bankz)' Official Teaser 2 (HYBE LABELS)\n\n\n6. Hearts2Hearts 하츠투하츠 'Pretty Please' MV (SMTOWN)\n\n\n7. 머쉬베놈 (MUSHVENOM) - 몰러유 (I don't know) Official MV (머쉬베놈)\n\n\n8. 2000년대 명곡 모음 💿 SG워너비, 씨야, 김범수, 이승철 ｜ 그 시절 감성을 되살리는 발라드 & 명곡 플레이리스트 (Lofi Shop)\n\n\n9. 먼데이 키즈, 이이경 – 운명 (2025) | (TEXTED) Destiny (2025) | 가사 (Lyrics) |... (Release - Topic)\n\n\n10. 야훼 (Live) (WELOVE - Topic)\n"}

[최신가요 실시간 인기차트] 2025년 9월 27일 4주차, 멜론차트 X, 종합차트, 차트둥이 노래모음 KPOP 플레이리스트 (Beautiful Life)
에스쿱스X민규 (SEVENTEEN) '5, 4, 3 (Pretty woman) (feat. Lay Bankz)' Official Teaser 2 (HYBE LABELS)
먼데이 키즈, 이이경 – 운명 (2025) | (TEXTED) Destiny (2025) | 가사 (Lyrics) |...

## 4. 메시지 생성 (기존 프롬프트)

프롬프트에는 `random.choice(result_tool)`, `result_attendence`, `user_command_dict.get('content')`와 가이드라인(도구 결과 사용, 완료 시 칭찬·미완료 시 격려, 자기결정성 이론, 따뜻하고 유머러스한 말투, 30단어 이내 한 줄)이 들어갑니다.

In [ ]:
#텍스트 전용 모델 로드하기
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
from pathlib import Path
import torch

# GPU 사용 가능 여부 확인 및 디바이스 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"사용할 디바이스: {device}")

# Tokenizer와 Model을 로컬에서 불러옴
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
base_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype="auto",
    device_map="auto",
    local_files_only=True
)

# LoRA 설정
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.5,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# PEFT 모델 구성
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

사용할 디바이스: cuda


trainable params: 4,374,528 || all params: 5,443,812,800 || trainable%: 0.0804


In [ ]:
def gemma_generate(prompt: str) -> str:
    model.eval()

    # --- 토큰/스페셜 처리 ---
    eos = tokenizer.convert_tokens_to_ids("<end_of_turn>")
    if eos is None or eos == tokenizer.unk_token_id:
        eos = tokenizer.eos_token_id
    pad = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id

    def _run(do_sample: bool, device) -> str:
        # 항상 모델과 같은 디바이스로 실행: 현재 A100을 사용 중
        inputs = tokenizer(prompt, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        gen_kwargs = dict(
            max_new_tokens=128,
            do_sample=do_sample,
            eos_token_id=eos,
            pad_token_id=pad,
            use_cache=True,
            top_p=0.9 if do_sample else None,
            temperature=0.7 if do_sample else None,
        )
        if do_sample:
            # top_k는 지원 안하면 빼기
            gen_kwargs.update(dict(temperature=0.7, top_p=0.8))

        with torch.no_grad():
            outputs = model.generate(**inputs, **gen_kwargs)

        input_len = inputs["input_ids"].shape[1]
        gen_ids = outputs[0, input_len:].clone()
        return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

    try:
        # 1) GPU + 샘플링
        return _run(True, next(model.parameters()).device)
    except RuntimeError as e1:
        msg = str(e1).lower()
        if "graph" in msg or "multinomial" in msg or "offset increment" in msg:
            try:
                # 2) GPU + 그리디(샘플링 끄기)
                return _run(False, next(model.parameters()).device)
            except Exception:
                pass
        # 3) CPU로 폴백 (샘플링 허용)
        try:
            model.to("cpu")
            return _run(True, "cpu")
        finally:
            # 사용 후 원래 디바이스로 복귀 (선택)
            try:
                model.to("cuda")
            except Exception:
                pass

In [ ]:
# GEMMA LoRA 기반 학습을 하지 않은 모델에서의 메세지 생성
import json
import random

user_command_dict = json.loads(user_command)

for i in range(5):
  followup_prompt = (
          f"<start_of_turn>user\n숙제 {result_attendence} 아이에게 1문장 동기부여 메세지 생성\n"
          f"[도구 결과] {random.choice(result_tool)}\n"
          f"[응원 메시지 생성 가이드라인]\n"
          f"1. 도구 결과를 메시지에 무조건 사용해줘 그리고 학생의 수업/과제 상황과 자연스럽게 연결해.\n"
          f"2. 숙제 완료이면 칭찬, 아니면 미완료이면 격려하는 표현을 사용해.\n"
          f"3. 자기결정성이론(Self-Determination Theory)에 따라 자율성, 유능감, 관계성을 북돋는 문장을 포함해.\n"
          f"4. 학생이 공감할 수 있도록 따뜻하고 유머러스한 말투로 작성해.\n"
          f"위의 정보를 바탕으로 학생에게 보낼 따뜻한 1인칭 응원 메시지를 30단어 이내의 한 줄로 작성해줘.: {user_command_dict.get('content')}<end_of_turn>\n"
          f"<start_of_turn>model\n"
  )

  final_response = gemma_generate(followup_prompt)
  print(f"최종 응답: {final_response}")

최종 응답: 와, 챌린지 54회차 인증 완료라니! 에스쿱스X민규처럼 멋진 결과 만들어가는 너의 자율성과 유능함에 감탄해! 든든한 관계 속에서 스스로 해내는 기쁨을 느껴봐! 😊
최종 응답: 와! 챌린지 54회차까지 꾸준히 참여하다니 정말 대단해! 임영웅 'ULSSIGU'처럼 멋진 나를 발견하고, 스스로 할 수 있다는 유능감을 느껴보자! 😊
최종 응답: 와, NMIXX의 'Blue Valentine'처럼 멋진 결과, 챌린지 54회차 인증 완료라니! 스스로 해낸 유능함에 자부심 느껴봐! 이 성취를 통해 앞으로 더 큰 목표를 향해 나아갈 수 있을 거야! 😊
최종 응답: 야훼(Live) 챌린지 54회차, 정말 멋진 성취야! 스스로 해낸 기쁨과 성취감을 느껴보렴. 이 맛에 계속 도전하고 싶어지는 거 같아! 😊 WELOVE!
최종 응답: 와, 챌린지 54회차 인증 완료라니! 스스로 해낸 성취감에 满满의 유능감을 느끼고 있겠네. 이 기운으로 다음 챌린지도 멋지게 해낼 수 있을 거야! 😊✨
